# Healthcare Liver Disease Decision Support — Milestone 3
**Participant:** Participant_M3_02  
**Project:** Healthcare Liver Disease Decision Support  
**Milestone:** 3  
**Dataset package:** `HealthCareData.zip`

## Objective
Predict liver disease status for educational decision support, explain model factors, and communicate risk responsibly.

## Milestone scope
neural network classifier; compare with best ML model on same test set; NLP on supplementary healthcare education text; document limitations.

> This is a synthetic sample submission for evaluator testing. Dataset files must be placed in the paths referenced below. Results are not pre-filled or fabricated.

## Neural network classification
Compare the neural network against the selected ML classifier using the same chronological-independent stratified holdout split. This is educational decision support only.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

df = pd.read_csv(Path("data")/"liver_patient_joined.csv")
y = df["Status"].map({1:1, 2:0})
X = df.drop(columns=["Status","ID"], errors="ignore")
cat = X.select_dtypes(include=["object","category"]).columns.tolist()
num = X.select_dtypes(include=np.number).columns.tolist()
prep = ColumnTransformer([
 ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
 ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))]),cat)
])
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
A=prep.fit_transform(Xtr); B=prep.transform(Xte)
ml=RandomForestClassifier(n_estimators=200,class_weight="balanced",random_state=42).fit(A,ytr)
nn=MLPClassifier(hidden_layer_sizes=(64,32),early_stopping=True,max_iter=250,random_state=42).fit(A,ytr)
for name, model in [("RandomForest",ml),("NeuralNetwork",nn)]:
 prob=model.predict_proba(B)[:,1]; pred=model.predict(B)
 print(name, "ROC-AUC",roc_auc_score(yte,prob),"recall",recall_score(yte,pred),"F1",f1_score(yte,pred))

## NLP activity
Load the supplied healthcare education text. Choose a task supported by available labels or use summarization/entity extraction as a qualitative exercise. Do not treat generated text as clinical advice.

In [ ]:
from pathlib import Path
text_df = pd.read_csv(Path("data")/"Healthcare_Education_Text.csv")
print(text_df.shape, text_df.columns.tolist())
display(text_df.head())